# Task 13 – RFM Analysis

Customer Behavior Analysis using Recency, Frequency, Monetary framework.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

In [ ]:
df = pd.read_csv("outputs/sales_data.csv")
df.head()

In [ ]:
df.info()
df.isnull().sum()

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

if "Revenue" not in df.columns:
    df["Revenue"] = df["Quantity"] * df["Unit_Price"]

df[["Order_Date", "Revenue"]].head()

In [ ]:
analysis_date = df["Order_Date"].max() + pd.Timedelta(days=1)
analysis_date

In [ ]:
rfm = df.groupby("Customer_ID").agg({
    "Order_Date": lambda x: (analysis_date - x.max()).days,
    "Order_ID": "count",
    "Revenue": "sum"
}).reset_index()

rfm.columns = ["Customer_ID", "Recency", "Frequency", "Monetary"]
rfm.head()

In [ ]:
rfm.describe()

In [ ]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1])
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5])
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels=[1,2,3,4,5])

rfm.head()

In [ ]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm.head()

In [ ]:
def rfm_segment(row):
    if row["R_Score"] >= "4" and row["F_Score"] >= "4" and row["M_Score"] >= "4":
        return "Champions"
    elif row["F_Score"] >= "4":
        return "Loyal Customers"
    elif row["R_Score"] <= "2":
        return "At Risk"
    else:
        return "Potential Customers"

rfm["Segment"] = rfm.apply(rfm_segment, axis=1)
rfm.head()

In [ ]:
rfm["Segment"].value_counts()

In [ ]:
plt.figure(figsize=(8,4))
rfm["Segment"].value_counts().plot(kind="bar")
plt.title("Customer Count by RFM Segment")
plt.tight_layout()
plt.show()

In [ ]:
segment_revenue = rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)
segment_revenue

In [ ]:
plt.figure(figsize=(8,4))
segment_revenue.plot(kind="bar")
plt.title("Revenue by RFM Segment")
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("outputs/plots", exist_ok=True)

rfm.to_csv("outputs/rfm_table.csv", index=False)
segment_revenue.reset_index().to_csv("outputs/rfm_segments_revenue.csv", index=False)

plt.figure(figsize=(8,4))
rfm["Segment"].value_counts().plot(kind="bar")
plt.tight_layout()
plt.savefig("outputs/plots/rfm_segment_count.png", dpi=200)
plt.close()

print("RFM outputs saved successfully")

## Task 13 – Conclusion

- Champions are the most valuable customers.
- Loyal customers purchase frequently and should be retained.
- At-risk customers require re-engagement strategies.
- RFM analysis enables targeted marketing decisions.